# M05A — Probabilistic Models: HMM & Naive Bayes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tulip-lab/pattern-classification-lab/blob/develop/M05-Parametric-Models/M05A-Probabilistic-Models-HMM-and-Naive-Bayes.ipynb)

**Pattern Classification practical · approximately 2 hours · M05 · CLO1, CLO2, CLO3**

This is the canonical student-facing practical for session `M05A`.
It was newly authored for the Pattern Classification Lab using concepts and
public migration references from `tulip-lab/flip01`: `F1A-03-HMM.ipynb`. No legacy
notebook cells or outputs are copied verbatim. The implementation uses current
public APIs and synthetic or scikit-learn-bundled data.

**Licence:** code cells are available under MPL-2.0; original narrative teaching
content is available under CC BY-NC-SA 4.0. Third-party works and datasets retain
their own terms. See the repository `LICENSE`, `CONTENT-LICENSE.md`, and
`NOTICE.md`.

## Overview

Naive Bayes models a fixed feature vector, whereas an HMM adds temporal dependence through latent states. This lab pairs a modern baseline classifier with a corrected, numerically stable Viterbi implementation adapted from the source material.

Work through the guided cells first, then complete the challenge in your own
copy. Keep your interpretation beside the code: a result without an argument
is not complete evidence.

## Learning Objectives

- Explain the conditional-independence assumptions of Naive Bayes and a first-order HMM.
- Train and evaluate a Gaussian Naive Bayes classifier.
- Implement Viterbi decoding in log space for a discrete HMM.
- Validate probability tables and decoded state sequences.

## Expected Output and Evidence

Retain the following evidence in your executed notebook:

- a held-out Gaussian Naive Bayes evaluation
- a decoded HMM state path and log-probability trace
- a before/after comparison for one explicit probabilistic assumption

Do not include credentials, private data, or another learner's work.

## Prerequisites

- Conditional probability
- Logarithms and products of probabilities
- Basic supervised classification

## Session Plan

| Time | Activity | Evidence |
|---:|---|---|
| 0–10 min | Orientation and diagnostic prompt | Initial prediction or sketch |
| 10–30 min | Background concepts and setup | Concept notes and reproducible environment |
| 30–75 min | Guided practice | Executed analysis with interpretations |
| 75–105 min | Student challenge | Independent model choice and evidence |
| 105–115 min | Testing and debugging | Passing checks and resolved issues |
| 115–120 min | Submission and reflection | Concise evidence-based reflection |

## Background Concepts

Naive Bayes assumes features are conditionally independent given the class: $p(x\mid C_k)=\prod_jp(x_j\mid C_k)$. Gaussian Naive Bayes estimates one mean and variance per class-feature pair.

An HMM describes an initial state distribution, a transition matrix, and an emission model. The Viterbi algorithm uses dynamic programming to recover the single most probable latent-state path. Log probabilities turn products into sums and reduce numerical underflow.

## Accessibility and Responsible Use

Every visual result in the guided path is paired with a printed numeric summary. Do not rely on colour alone in your challenge evidence: add labels, line styles, markers, or a compact table as appropriate. Add concise alt text when exporting figures for another format. Use only public, approved, or synthetic data.

Follow the collaboration, acknowledgement, and AI-use rules published by your current offering; this institution-neutral practical does not define those rules.

## Setup

Run the next cell before starting. All datasets are bundled with scikit-learn or generated locally; no download is required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

## Guided Practice

Follow the sequence and pause at each interpretation prompt before moving on.

### 1. Gaussian Naive Bayes on Iris

Each row is treated independently. The model estimates feature distributions from the training split only.

In [ ]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, stratify=iris.target, random_state=RANDOM_STATE
)
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_predictions = nb_model.predict(X_test)
nb_probabilities = nb_model.predict_proba(X_test)
print(f"Naive Bayes held-out accuracy: {accuracy_score(y_test, nb_predictions):.3f}")
ConfusionMatrixDisplay.from_predictions(y_test, nb_predictions, display_labels=iris.target_names, cmap="Purples")
plt.show()

### 2. Define a small discrete HMM

Rows must sum to one. States are latent study modes; observations are visible behaviours. These labels are illustrative, not claims about people.

In [ ]:
states = np.array(["focused", "transitioning", "resting"])
observations = np.array(["typing", "walking", "phone"])
initial = np.array([0.60, 0.25, 0.15])
transition = np.array([
    [0.70, 0.20, 0.10],
    [0.25, 0.50, 0.25],
    [0.15, 0.25, 0.60],
])
emission = np.array([
    [0.75, 0.10, 0.15],
    [0.20, 0.60, 0.20],
    [0.10, 0.15, 0.75],
])
sequence = np.array([0, 0, 1, 1, 2, 2, 0, 0], dtype=int)
print("Observed sequence:", observations[sequence])

### 3. Viterbi decoding in log space

Back-pointers must use an integer dtype; this fixes a fragile detail in the original implementation.

In [ ]:
def viterbi_log(initial, transition, emission, observed):
    n_states = len(initial)
    n_steps = len(observed)
    delta = np.full((n_steps, n_states), -np.inf)
    backpointer = np.zeros((n_steps, n_states), dtype=int)
    delta[0] = np.log(initial) + np.log(emission[:, observed[0]])

    for t in range(1, n_steps):
        for current in range(n_states):
            candidates = delta[t - 1] + np.log(transition[:, current])
            backpointer[t, current] = np.argmax(candidates)
            delta[t, current] = candidates[backpointer[t, current]] + np.log(emission[current, observed[t]])

    path = np.zeros(n_steps, dtype=int)
    path[-1] = np.argmax(delta[-1])
    for t in range(n_steps - 2, -1, -1):
        path[t] = backpointer[t + 1, path[t + 1]]
    return path, delta, backpointer

state_path, delta, backpointer = viterbi_log(initial, transition, emission, sequence)
for observation, state in zip(observations[sequence], states[state_path]):
    print(f"{observation:>8} -> {state}")

### 4. Compare what the two models condition on

Naive Bayes classifies independent rows using features. Viterbi decodes a joint sequence because each state decision depends on the previous state path.

In [ ]:
print("Naive Bayes probability shape:", nb_probabilities.shape)
print("Viterbi dynamic-programming table shape:", delta.shape)
print("Decoded path log probability:", delta[-1, state_path[-1]].round(3))

## Student Challenge

Change one modelling assumption and test its effect: either make state persistence stronger in the transition matrix, or make one observation less diagnostic in the emission matrix. Decode the same observations before and after, then explain every changed state.

In [ ]:
modified_transition = transition.copy()
modified_emission = emission.copy()

# TODO: change exactly one complete row, ensuring it remains a probability distribution.
modified_path, _, _ = viterbi_log(initial, modified_transition, modified_emission, sequence)
print("original:", states[state_path])
print("modified:", states[modified_path])

### Challenge success criteria

- Exactly one stated assumption is changed.
- All probability rows remain non-negative and sum to one.
- The before/after paths are compared at each time step.
- The explanation refers to transition or emission evidence, not only the output.

## Testing and Debugging

Run these checks after the guided practice. If one fails, inspect shapes, label order, random seeds, and whether preprocessing was fitted only on training data.

In [ ]:
def validate_probability_rows(name, values):
    if np.any(values < 0) or not np.allclose(values.sum(axis=1), 1.0):
        raise ValueError(f"{name} rows must be non-negative and sum to one.")

# Exercise malformed probability-table behaviour.
invalid_transition = transition.copy()
invalid_transition[0, 0] = 1.2
try:
    validate_probability_rows("transition", invalid_transition)
except ValueError as error:
    print("Expected probability-table failure:", error)
else:
    raise AssertionError("An invalid transition table should fail clearly.")

validate_probability_rows("transition", transition)
validate_probability_rows("emission", emission)
print("Edge check: the one-step observation sequence can also be decoded:",
      states[viterbi_log(initial, transition, emission, sequence[:1])[0]])

In [ ]:
assert np.allclose(initial.sum(), 1.0)
assert np.allclose(transition.sum(axis=1), 1.0)
assert np.allclose(emission.sum(axis=1), 1.0)
assert np.allclose(nb_probabilities.sum(axis=1), 1.0)
assert state_path.dtype.kind in "iu" and len(state_path) == len(sequence)
print("M05A guided-practice checks passed.")

## Extension Task

Implement the forward algorithm and compare the total sequence log-likelihood with the Viterbi path log-probability. Explain why the former cannot be smaller.

## Submission and Reflection

Submit your completed notebook (or an HTML export) with outputs visible. Include: 

1. the challenge code and result;
2. one figure or compact results table;
3. a 150–250 word reflection answering: **What changed, why did it change, and what evidence supports your explanation?**

Follow your offering's published collaboration, acknowledgement, and AI-use rules. You remain responsible for checking and explaining all submitted work.

## References

- Source: `F1A-03-HMM.ipynb` (Viterbi example modernised and corrected).
- Rabiner (1989), “A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition.”
- scikit-learn User Guide: [Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)